# 🎓 Day 1 · Session 5
# 05B · Build Your First RAG Chatbot
## A working knowledge bot using chunks, embeddings, retrieval and grounded prompting

## 🎯 Learning Objectives

- Build a simple RAG pipeline
- Retrieve relevant chunks
- Generate grounded answers with source references

## 🔧 Setup

Expected `.env` file in the same folder as this notebook:

```env
OPENAI_API_KEY=sk-your-key-here
```

In [ ]:
# Uncomment if required
# %pip install openai python-dotenv pandas numpy scikit-learn pypdf chromadb langchain langchain-openai langchain-community langchain-chroma

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
import pandas as pd
import numpy as np
from openai import OpenAI

In [ ]:
env_path = Path.cwd() / ".env"

print("Current working directory:", Path.cwd())
print("Looking for .env at:", env_path)
print("Env exists:", env_path.exists())

load_dotenv(env_path, override=True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("Dotenv keys found:", list(dotenv_values(env_path).keys()))
    raise ValueError("OPENAI_API_KEY not found. Please check your .env file.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully.")
print("Key starts with:", api_key[:10], "...")

In [ ]:
def ask_openai(prompt, model="gpt-4o-mini", temperature=0.2, system_prompt="You are a helpful academic assistant.", max_tokens=700):
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

# 1️⃣ Build Your First RAG Chatbot

In this notebook, we build a working RAG pipeline.

Pipeline:

```text
Sample Documents → Chunks → Embeddings → ChromaDB → Retriever → Prompt → LLM Answer
```

In [ ]:
sample_docs = [
    {
        "source": "department_handbook.txt",
        "text": '''
ABC Institute of Technology - Department Handbook

M.Tech CSE Fee Structure:
The tuition fee for M.Tech CSE is Rs. 50,000 per semester.
The programme duration is 4 semesters.
GATE-qualified students are eligible for a 50% tuition scholarship.

Attendance Rule:
Students must maintain 75% attendance to be eligible for final examinations.
'''
    },
    {
        "source": "course_catalog.txt",
        "text": '''
Course Catalog

NLP Elective:
The NLP elective is offered in Semester 3.
Prerequisites: Machine Learning and Python Programming.
Faculty Coordinator: Dr. Kavitha Raman.

Deep Learning Elective:
Offered in Semester 4.
Prerequisites: Linear Algebra, Python Programming and Machine Learning.
'''
    },
    {
        "source": "lab_manual.txt",
        "text": '''
AI Lab Manual

Lab 1: Python for AI
Lab 2: Data preprocessing
Lab 3: Linear regression
Lab 4: Classification
Lab 5: Neural networks
Lab 6: Model evaluation

Students must submit lab records every Friday.
'''
    }
]

print("Number of documents:", len(sample_docs))

# 2️⃣ Create Chunks

In [ ]:
def chunk_documents(docs, chunk_size=400, overlap=80):
    chunks = []

    for doc in docs:
        text = doc["text"]
        source = doc["source"]
        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "source": source,
                "start": start,
                "end": end
            })

            start = end - overlap

    return chunks

chunks = chunk_documents(sample_docs)

print("Number of chunks:", len(chunks))
for i, chunk in enumerate(chunks, 1):
    print(f"Chunk {i} | Source: {chunk['source']}")
    print(chunk["text"][:250])
    print("-" * 80)

# 3️⃣ Create Embeddings

In [ ]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

print("Embeddings created.")
print("Embedding dimension:", len(chunks[0]["embedding"]))

# 4️⃣ Build a Simple Retriever

In [ ]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query, chunks, top_k=3):
    query_embedding = get_embedding(query)

    scored = []
    for chunk in chunks:
        score = cosine_similarity(query_embedding, chunk["embedding"])
        scored.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "score": score
        })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]

results = retrieve("What are the prerequisites for NLP?", chunks)

pd.DataFrame(results)

# 5️⃣ Build RAG Answer Function

In [ ]:
def rag_answer(question, chunks, top_k=3):
    retrieved = retrieve(question, chunks, top_k=top_k)

    context = "\n\n".join([
        f"Source: {item['source']}\nContent: {item['text']}"
        for item in retrieved
    ])

    prompt = f'''
You are a helpful academic assistant.

Answer the question using ONLY the context below.
If the answer is not present in the context, say:
"I don't have this information in the provided knowledge base."

Mention the source file used.

Context:
{context}

Question:
{question}
'''

    answer = ask_openai(prompt)

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved
    }

response = rag_answer("What are the prerequisites for NLP?", chunks)

print(response["answer"])

# 6️⃣ Inspect Retrieval

In [ ]:
for i, item in enumerate(response["retrieved_chunks"], 1):
    print(f"--- Retrieved Chunk {i} ---")
    print("Source:", item["source"])
    print("Score:", round(item["score"], 4))
    print(item["text"])
    print()

# 7️⃣ Test In-Scope and Out-of-Scope Questions

In [ ]:
test_questions = [
    "What is the M.Tech CSE fee?",
    "Who is the faculty coordinator for NLP?",
    "When should students submit lab records?",
    "What is the hostel fee?",
    "Who is the principal of the college?"
]

for q in test_questions:
    print("\nQUESTION:", q)
    result = rag_answer(q, chunks)
    print("ANSWER:", result["answer"])
    print("-" * 100)

# 8️⃣ Optional: Use ChromaDB

In [ ]:
# Optional ChromaDB example.
# Uncomment after installing:
# %pip install chromadb langchain langchain-openai langchain-chroma

RUN_CHROMA_DEMO = False

if RUN_CHROMA_DEMO:
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_chroma import Chroma
    from langchain_core.documents import Document

    docs = [
        Document(page_content=chunk["text"], metadata={"source": chunk["source"]})
        for chunk in chunks
    ]

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embeddings,
        persist_directory="./chroma_day1_session5"
    )

    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    docs_found = retriever.invoke("What are the prerequisites for NLP?")

    for d in docs_found:
        print(d.metadata, d.page_content[:300])
else:
    print("Skipping ChromaDB demo. Set RUN_CHROMA_DEMO=True after installing dependencies.")

# 9️⃣ Simple Chat Loop

In [ ]:
def ask_knowledge_bot(question):
    result = rag_answer(question, chunks)
    return result["answer"]

print(ask_knowledge_bot("Explain the NLP elective details."))

# 🧪 Lab

Replace `sample_docs` with your own:

- syllabus
- lab manual
- department FAQ
- course notes

Then test:
1. 3 factual questions
2. 2 out-of-scope questions
3. 1 ambiguous question

# ✅ Summary

You built a working RAG chatbot using chunks, embeddings, retrieval and grounded prompting.